In [1]:
#to prevent colab to automatically disconnect
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button")
   if (btn != null){
     console.log("Click colab-connect-button");
     btn.click()
     }

   btn = document.getElementById('ok')
   if (btn != null){
     console.log("Click reconnect");
     btn.click()
     }
  }

setInterval(ClickConnect,60000)
'''))

print("Done.")

<IPython.core.display.Javascript object>

Done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Step 1:Importing Libraries and Installing Dependencies**

In [3]:
pip install keras-self-attention

  Preparing metadata (setup.py) ... done
  Created wheel for keras-self-attention: filename=keras_self_attention-0.51.0-py3-none-any.whl size=18895 sha256=4f3f0ea5a2b1ff8f59a4dc17fb9c3e6ec76badf70ba9747a05470204fd5136b4
  Stored in directory: /root/.cache/pip/wheels/46/f9/96/709295c836133071c12a300729fed4027757f889c01695feea
Successfully built keras-self-attention


In [4]:
pip install tensorflow

In [5]:
# For data processing
import numpy as np
import math
from math import sqrt

# For data processing and manipulation
import pandas as pd
import csv

# For date calculations
import datetime
import time

# For ploting data
import IPython
import IPython.display

import itertools
from itertools import cycle
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# For checking path
import os , gc
import csv
import json


from scipy.stats import hmean

from sklearn.metrics import mean_squared_error, mean_absolute_error

#tensorflow libs
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping , Callback
import tensorflow as tf
from tensorflow.keras import backend as K
from keras import backend

from tensorflow.keras.layers import *
from tensorflow.keras.layers import Dense , LSTM ,Dropout , PReLU , RepeatVector ,TimeDistributed, Attention,LayerNormalization,Add
from tensorflow.keras.models import Sequential,load_model
from tensorflow.keras.utils import to_categorical , plot_model
from tensorflow.keras import regularizers, constraints, initializers, activations
#from keras.layers.recurrent import Recurrent, _time_distributed_dense
from tensorflow.keras.layers import SimpleRNN as Recurrent
#from tensorflow.compat.v1.keras.layers import RNN

from tensorflow.keras.layers import InputSpec

from keras_self_attention import SeqSelfAttention
from tensorflow.keras.layers import Concatenate

from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Bidirectional, TimeDistributed, LayerNormalization, Add
from tensorflow.keras import layers


tf.get_logger().setLevel('ERROR')
mpl.rcParams['figure.figsize'] = (8, 6)
mpl.rcParams['axes.grid'] = False

# **Step 2: Loading Dataset**

In [6]:
model_save_path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/Trained_model_Files'
dataset_path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/2.Preparing Dataset to feed Model/patrol_wise_crime_dataset'

files = os.listdir(dataset_path)
files

['Central_1.csv',
 'Rampart_2.csv',
 'Southwest_3.csv',
 'Hollenbeck_4.csv',
 'Harbor_5.csv',
 'Hollywood_6.csv',
 'Wilshire_7.csv',
 'West LA_8.csv',
 'Van Nuys_9.csv',
 'West Valley_10.csv',
 'Northeast_11.csv',
 '77th Street_12.csv',
 'Newton_13.csv',
 'Pacific_14.csv',
 'N Hollywood_15.csv',
 'Foothill_16.csv',
 'Devonshire_17.csv',
 'Southeast_18.csv',
 'Mission_19.csv',
 'Olympic_20.csv',
 'Topanga_21.csv']

In [7]:
#loading dataset
patrol_divisons = {}
dataset = {}
for file in files:
  name = file.split('_')[0]
  id = file.split('_')[1].split('.')[0]
  patrol_divisons[int(id)] = name
  dataset[int(id)] = pd.read_csv(os.path.join(dataset_path,file))

patrol_divisons = dict(sorted(patrol_divisons.items()))
patrol_divisons

{1: 'Central',
 2: 'Rampart',
 3: 'Southwest',
 4: 'Hollenbeck',
 5: 'Harbor',
 6: 'Hollywood',
 7: 'Wilshire',
 8: 'West LA',
 9: 'Van Nuys',
 10: 'West Valley',
 11: 'Northeast',
 12: '77th Street',
 13: 'Newton',
 14: 'Pacific',
 15: 'N Hollywood',
 16: 'Foothill',
 17: 'Devonshire',
 18: 'Southeast',
 19: 'Mission',
 20: 'Olympic',
 21: 'Topanga'}

In [8]:
columns_in_dataset = dataset[1].columns
columns_in_dataset

Index(['Unnamed: 0', 'datetime', 'p_id', '1', '2', '3', '4', '5', '6', '7',
       '8', 'group 0', 'count', 'day sin', 'day cos', 'week sin', 'week cos',
       'year sin', 'year cos'],
      dtype='object')

In [9]:
dataset[1].head()

,Unnamed: 0,datetime,p_id,1,2,3,4,5,6,7,8,group 0,count,day sin,day cos,week sin,week cos,year sin,year cos
0,0,2010-01-01 00:00:00,1,6.0,1.0,0.0,3.0,0.0,0.0,0.0,1.0,0.0,11.0,-4.416858e-12,1.000000e+00,0.781831,0.623490,0.005161,0.999987
1,1,2010-01-01 03:00:00,1,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,7.071068e-01,7.071068e-01,0.846724,0.532032,0.007311,0.999973
2,2,2010-01-01 06:00:00,1,0.0,0.0,0.0,12.0,1.0,0.0,0.0,1.0,0.0,14.0,1.000000e+00,6.980203e-12,0.900969,0.433884,0.009461,0.999955
3,3,2010-01-01 09:00:00,1,2.0,0.0,0.0,13.0,1.0,0.0,1.0,0.0,0.0,17.0,7.071068e-01,-7.071068e-01,0.943883,0.330279,0.011612,0.999933
4,4,2010-01-01 12:00:00,1,0.0,2.0,0.0,7.0,0.0,0.0,0.0,1.0,0.0,10.0,9.543547e-12,-1.000000e+00,0.974928,0.222521,0.013762,0.999905


In [10]:
print(dataset[1]["count"].max())
print(dataset[1]["count"].min())

69.0
0.0


Actual Crime Code :                                                                               ASSAULT': 0,
 'BURGLARY': 1,
 'CRIMINAL TRESPASS': 2,
 'DECEPTIVE PRACTICE': 3,
 'DRUG/NARCOTIC': 4,
 'HOMICIDE': 5,
 'HUMAN TRAFFICKING': 6,
 'INTERFERENCE WITH PUBLIC OFFICER': 7,
 'KIDNAPPING': 8,
 'LARCENY/THEFT': 9,
 'OTHER OFFENSES': 10,
 'PROSTITUTION': 11,
 'PUBLIC PEACE VIOLATION': 12,
 'ROBBERY': 13,
 'SEX OFFENSE': 14,
 'WEAPONS VIOLATION': 15

Grouped Crime Codes : [[0],[1],[2],[9],[10],[13],[14],[15],[3,4,5,6,7,8,11,12]]

# **Step 3: Converting to Time Series data**

**Utility Functions**

In [11]:
def train_test_val_split(dataset):
  columns_indices = {name:i for i,name in enumerate(dataset.columns)}
  n = len(dataset)

  #splitting dataset
  training_set = dataset[:int(n*0.7)]
  validation_set = dataset[int(n*0.7):int(n*0.9)]
  test_set = dataset[int(n*0.9):]

  num_features = dataset.shape[1]
  return training_set, validation_set, test_set, num_features, columns_indices

In [12]:
#@title
class WindowGenerator():
    '''
    WindowGenerator Class
    1. Split windows of features into a (features, labels) pairs.
    2. Plot the content of the resulting windows.
    3. Efficiently generate batches of these windows from the training, evaluation, and test data, using tf.data.Datasets S.
    '''
    def __init__(self, input_width, label_width, shift,
               train_df, val_df, test_df,
               label_columns=None , shuffle=False , batch_size = 64):
        '''
        The __init__ method includes all the necessary logic for the input and label indices.
        Input:
            input_width : input width / window size
            label_width : output width
            shift : size of window shifting forward
            train_df : train dataset
            val_df : validation dataset
            test_df : test dataset
            label_columns ( Default = None) : Label Columns
            shuffle ( Default = False) : weather to shuffle data
            batch_size (Default = 64) : Batch Size
        Output: None
        Example :
            w2 = WindowGenerator(input_width=6, label_width=1, shift=1,
                     label_columns=['count'])
            w2

        '''
        # Store the raw data.
        self.train_df = train_df
        self.val_df = val_df
        self.test_df = test_df
        self.shuffle = shuffle
        self.batch_size = batch_size

        # Work out the label column indices.
        self.label_columns = label_columns
        if label_columns is not None:
            self.label_columns_indices = {name: i for i, name in
                                        enumerate(label_columns)}

        self.column_indices = {name: i for i, name in
                            enumerate(train_df.columns)}


        # Work out the window parameters.
        self.input_width = input_width
        self.label_width = label_width
        self.shift = shift

        self.total_window_size = input_width + shift

        self.input_slice = slice(0, input_width) #(start , stop)
        self.input_indices = np.arange(self.total_window_size)[self.input_slice]

        self.label_start = self.total_window_size - self.label_width
        self.labels_slice = slice(self.label_start, None)
        self.label_indices = np.arange(self.total_window_size)[self.labels_slice]

    def __repr__(self):
        return '\n'.join([
            f'Total window size: {self.total_window_size}',
            f'Input indices: {self.input_indices}',
            f'Label indices: {self.label_indices}',
            f'Label column name(s): {self.label_columns}'])

    def split_window(self, features):
        '''
        Given a list consecutive inputs, the split_window method will convert them to a window of inputs and a window of labels.

        Input:
            features : Stack of array of datas , used for splitting data to inputs and labels
        Output:
            inputs : nd Array splitted as input_width
            labesl : nd Array splitted as label_width
        Example :
        # Stack three slices, the length of the total window:
            example_window = tf.stack([np.array(train_df[:w2.total_window_size]),
                           np.array(train_df[100:100+w2.total_window_size]),
                           np.array(train_df[200:200+w2.total_window_size])])


            example_inputs, example_labels = w2.split_window(example_window)

            print('All shapes are: (batch, time, features)')
            print(f'Window shape: {example_window.shape}')
            print(f'Inputs shape: {example_inputs.shape}')
            print(f'labels shape: {example_labels.shape}')
        '''
        inputs = features[:, self.input_slice, :]
        labels = features[:, self.labels_slice, :]
        #taking only the labels that are presentin the label_columns
        if self.label_columns is not None:
            labels = tf.stack([labels[:, :, self.column_indices[name]] for name in self.label_columns],axis=-1)

        # Slicing doesn't preserve static shape information, so set the shapes
        # manually. This way the `tf.data.Datasets` are easier to inspect.
        inputs.set_shape([None, self.input_width, None])
        labels.set_shape([None, self.label_width, None])

        return inputs, labels

    def plot(self, model=None, plot_col='count', max_subplots=3):
        '''
            plot method that allows a simple visualization of the split window,
            Input:
                model (Default=None) : tensorflow model to evaluate
                plot_col ( Default = 'count') : Name of column to evaluate
                max_subplots ( Default = 3) : Maximum Number of subplotting
            Output:
                None
            Example:
                w2.plot()
                w2.plot(plot_col=0) # label wont be shown as w2 config has only column , count

        '''
        inputs, labels = self.example
        plt.figure(figsize=(12, 8))
        plot_col_index = self.column_indices[plot_col]
        max_n = min(max_subplots, len(inputs))
        for n in range(max_n):
            plt.subplot(max_n, 1, n+1)
            plt.ylabel(f'{plot_col} [normed]')
            plt.plot(self.input_indices, inputs[n, :, plot_col_index],label='Inputs', marker='.', zorder=-10)

            if self.label_columns:
                label_col_index = self.label_columns_indices.get(plot_col, None)
            else:
                label_col_index = plot_col_index

            if label_col_index is None:
                continue

            plt.scatter(self.label_indices, labels[n, :, label_col_index],edgecolors='k', label='Labels', c='#2ca02c', s=64)
            if model is not None:
                predictions = model(inputs)
                plt.scatter(self.label_indices, predictions[n, :, label_col_index],
                        marker='X', edgecolors='k', label='Predictions',c='#ff7f0e', s=64)
            if n == 0:
                plt.legend()
        plt.xlabel('Time [h]')

    def make_dataset(self, data):
        '''
        make_dataset method will take a time series DataFrame and convert it to a
            tf.data.Dataset of (input_window, label_window) pairs using the preprocessing.timeseries_dataset_from_array function.
        Input:
            data :  Input data to transform into (input_window , label_window)
        Output:
            ds : transformed dataset
        '''
        data = np.array(data, dtype=np.float32)
        ds = tf.keras.preprocessing.timeseries_dataset_from_array(
            data=data,
            targets=None,
            sequence_length=self.total_window_size,
            sequence_stride=1,
            shuffle=self.shuffle,
            batch_size=self.batch_size,)
        ds = ds.map(self.split_window)
        return ds


    def create_dataset2(self , map_df , reshape=True):
      x = []
      y = []
      for res in iter(map_df):
        inputs, labels = res
        if(len(inputs)==64):
          x.append(inputs)
          y.append(labels)

      x = np.array(x)
      y = np.array(y)
      if(reshape):
        x = x.reshape(-1, x.shape[-2] , x.shape[-1])
        y = y.reshape(-1 , y.shape[-2] , y.shape[-1])
      return x , y

    '''
    properties for accessing  training, validation and test data as tf.data.Datasets using the above make_dataset method.
    Also a standard example batch for easy access and plotting
    '''

    @property
    def train(self):
        return self.make_dataset(self.train_df)

    @property
    def val(self):
        return self.make_dataset(self.val_df)

    @property
    def test(self):
        return self.make_dataset(self.test_df)


    @property
    def example(self):
        """Get and cache an example batch of `inputs, labels` for plotting."""
        result = getattr(self, '_example', None)
        if result is None:
            # No example batch was found, so get one from the `.train` dataset
            result = next(iter(self.train))
            # And cache it for next time
            self._example = result
        return result

In [13]:
def create_data(train , test , val , columns):
    '''
    Create dataset from main train , test , val with given columns
    '''
    if(columns==None):
        columns = train.columns
    new_train = train[columns]
    new_test = test[columns]
    new_val = val[columns]
    return new_train , new_test , new_val

In [14]:
def save_history(history , path):
    # convert the history.history dict to a pandas DataFrame:
    hist_df = pd.DataFrame(history.history)
    # or save to csv:
    hist_csv_file = path
    with open(hist_csv_file, mode='w') as f:
        hist_df.to_json(f)

def get_history(path):
	with open(path) as json_file:
		data = json.load(json_file)
		return data

def save_model_weights(model , path):
  model.save_weights(path)


Functions for Model Compiling and Fitting

In [15]:
def compileModel(model):

        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
          initial_learning_rate=1e-4,
          decay_steps=1000,
          decay_rate=0.96,
          staircase=True,
      )
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

        model.compile(loss=tf.losses.MeanAbsoluteError(),optimizer = optimizer,
                   metrics =[ tf.keras.metrics.RootMeanSquaredError(name='rmse'),
                              tf.keras.metrics.MeanAbsoluteError(name='mae')])
        return model

MAX_EPOCHS = 30

def fit(model,ROOTPATH, modelPath , historyPath , window=None, name=None , patience = 5):
    '''
    Compile and fit a model
    '''

    modelPathPar = os.path.join(ROOTPATH,modelPath)
    historyPathPar = os.path.join(ROOTPATH,historyPath)

    #Creating a file to store model and it's history
    if (name!=None):
        model_path = os.path.join(modelPathPar,name+".keras")
        history_path = os.path.join(historyPathPar,name+".json")

        if not os.path.exists(modelPathPar):
            os.makedirs(modelPathPar)
        if not os.path.exists(historyPathPar):
            os.makedirs(historyPathPar)

    if (name!=None and os.path.exists(model_path)):
      print("Loaded Pre Trained Model")
      modelOld = tf.keras.models.load_model(model_path)
      model.set_weights(modelOld.get_weights())
      del modelOld
      history = get_history(history_path)
      return model , history



    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=patience)

    history = model.fit(window.train, epochs=MAX_EPOCHS,batch_size = 64,validation_data=window.val,
                        callbacks=[early_stopping],verbose=1)


    if(name!=None):
      model.save(model_path)
      save_history(history , history_path)

    return model , history.history

Function to save  metric values

In [16]:
def build_metrics_dataframe(test_metrics_dict, val_metrics_dict, patrol_divisions_dict):
    """
    Converts performance dictionaries into a structured DataFrame.

    Parameters:
        test_metrics_dict (dict): Dict of test evaluation results {division_id: [loss, rmse, mae]}
        val_metrics_dict (dict): Dict of val evaluation results {division_id: [loss, rmse, mae]}
        patrol_divisions_dict (dict): Dict mapping division_id to division name

    Returns:
        pd.DataFrame: DataFrame with division names as rows and metrics as columns
    """
    rows = []

    for div_id in test_metrics_dict.keys():
        div_name = patrol_divisions_dict[int(div_id)]
        test = test_metrics_dict[div_id]
        val = val_metrics_dict[div_id]

        rows.append({
            "Division": div_name,
            "test rmse": test[1],
            "test mae": test[2],
            "val rmse": val[1],
            "val mae": val[2]
        })

    df = pd.DataFrame(rows)
    df.set_index("Division", inplace=True)
    return df


In [17]:
general_indexs = ['1', '2', '3', '4', '5', '6', '7','8', 'count',
           'day sin', 'day cos', 'week sin', 'week cos',
           'year sin', 'year cos', 'group 0']

x_col = ['day sin' , 'day cos' , 'year sin' , 'year cos' , 'week cos' , 'week sin' ,'datetime']

def generate_window(df_now, ret_test = 0):
    train_df , val_df , test_df , num_features_df , column_indices_df = train_test_val_split(df_now)
    train_df , test_df , val_df = create_data(train_df , test_df , val_df , general_indexs)
    y_col = []

    #storing label columns
    for i in train_df.columns:
        if (i in x_col):
            continue
        y_col.append(i)

    #creating window generator object
    wide_window_all = WindowGenerator(train_df=train_df, test_df=test_df , val_df=val_df,
        input_width=24, label_width=24, shift=1,
        label_columns=y_col)

    if (ret_test == 1):
      return wide_window_all, test_df
    else:
      return wide_window_all


**1.Multi Head Attention Bi LSTM**

In [ ]:
inputs = keras.Input(shape=(24,16), name="input_bi_lstm")

# First BiLSTM
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, activation='swish'), name="bilstm1")(inputs)
x = layers.Dropout(0.3)(x)

# Multi-Head Attention Layer (Self-Attention)
attn_output = layers.MultiHeadAttention(num_heads=4, key_dim=32, name="multihead_attention_bilstm")(x, x)
attn_output = layers.Dropout(0.3)(attn_output)
attn_output = layers.LayerNormalization(epsilon=1e-6)(x + attn_output)  # Residual

# Feed-Forward Layer after attention
ff = layers.TimeDistributed(layers.Dense(64, activation='swish'), name="ff1")(attn_output)
ff = layers.Dropout(0.3)(ff)

# Project attention output to match ff shape for residual connection
attn_proj = layers.TimeDistributed(layers.Dense(64), name="attn_proj_bilstm")(attn_output)
ff_output = layers.LayerNormalization(epsilon=1e-6)(attn_proj + ff)

# Final Dense output layer (per time step)
output = layers.TimeDistributed(layers.Dense(10), name="output_layer_bilstm")(ff_output)

# Define the model
mh_attn_bi_lstm = keras.Model(inputs, output, name="BiLSTM_MHAttention_Residual")
mh_attn_bi_lstm.summary()


Model: "BiLSTM_MHAttention_Residual"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_bi_lstm       │ (None, 24, 16)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm1             │ (None, 24, 256)   │    148,480 │ input_bi_lstm[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 24, 256)   │          0 │ bilstm1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multihead_attentio… │ (None, 24, 256)   │    131,712 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 24, 256)   │          0 │ multihead_attent… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 24, 256)   │          0 │ dropout[0][0],    │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 24, 256)   │        512 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff1                 │ (None, 24, 64)    │     16,448 │ layer_normalizat… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_proj_bilstm    │ (None, 24, 64)    │     16,448 │ layer_normalizat… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 24, 64)    │          0 │ ff1[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 24, 64)    │          0 │ attn_proj_bilstm… │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 24, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_layer_bilstm │ (None, 24, 10)    │        650 │ layer_normalizat… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 314,378 (1.20 MB)

 Trainable params: 314,378 (1.20 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
val_performance_of_mh_attn_bi_lstm = {}
performance_of_mh_attn_bi_lstm = {}
histories_of_mh_attn_bi_lstm = {}
mh_attn_bi_lstm_model = compileModel(mh_attn_bi_lstm)
save_path = os.path.join(model_save_path,"Multi Head Attention Bi LSTM")
for x in patrol_divisons.keys():
        wide_window_all = generate_window(dataset[x])
        print(x,patrol_divisons[x])
        x = str(x)
        trained_mh_attn_bi_lstm_model , history  = fit(
            mh_attn_bi_lstm_model,save_path, "mh_attn_bi_lstm_model_files" , "mh_attn_bi_lstm_history_files" , name='mh_attn_bi_lstm_all_feature_'+x , window=wide_window_all)
        histories_of_mh_attn_bi_lstm['bd_'+x] = history
        val_performance_of_mh_attn_bi_lstm[x] = trained_mh_attn_bi_lstm_model.evaluate(wide_window_all.val)
        performance_of_mh_attn_bi_lstm[x] = trained_mh_attn_bi_lstm_model.evaluate(wide_window_all.test)


1 Central
Epoch 1/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 44s 51ms/step - loss: 0.7006 - mae: 0.7006 - rmse: 1.0729 - val_loss: 0.7517 - val_mae: 0.7517 - val_rmse: 1.3319
Epoch 2/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - loss: 0.5009 - mae: 0.5009 - rmse: 0.8664 - val_loss: 0.7467 - val_mae: 0.7467 - val_rmse: 1.2006
Epoch 3/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - loss: 0.4694 - mae: 0.4694 - rmse: 0.8216 - val_loss: 0.6558 - val_mae: 0.6558 - val_rmse: 1.1401
Epoch 4/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - loss: 0.4264 - mae: 0.4264 - rmse: 0.7815 - val_loss: 0.6264 - val_mae: 0.6264 - val_rmse: 1.0958
Epoch 5/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - loss: 0.3949 - mae: 0.3949 - rmse: 0.7358 - val_loss: 0.4830 - val_mae: 0.4830 - val_rmse: 0.8671
Epoch 6/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - loss: 0.3424 - mae: 0.3424 - rmse: 0.6464 - val_loss: 0.4715 - val_mae: 0.4715 - val_rmse: 0.8535
Epoch 7/30
483/483 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - loss: 0.3095 

In [ ]:
# Call the function after training is done
mh_attn_bi_lstm_results_df = build_metrics_dataframe(
     performance_of_mh_attn_bi_lstm,
    val_performance_of_mh_attn_bi_lstm,
    patrol_divisons
)

csv_path = os.path.join(save_path,"mh_attn_bi_lstm_metrics_1.csv")
mh_attn_bi_lstm_results_df.to_csv(csv_path)


In [ ]:
mh_attn_bi_lstm_results_df.head(21)

,test rmse,test mae,val rmse,val mae
Division,,,,
Central,0.747138,0.231611,0.533995,0.213056
Rampart,0.442736,0.114495,0.365743,0.120840
Southwest,0.384467,0.112412,0.503740,0.128599
Hollenbeck,0.213087,0.056105,0.276496,0.076320
Harbor,0.240912,0.058873,0.285419,0.075972
Hollywood,0.297281,0.082892,0.362755,0.115264
Wilshire,0.271196,0.073144,0.319655,0.095985
West LA,0.232333,0.056557,0.302925,0.087338
Van Nuys,0.220680,0.054640,0.272708,0.074478


**Finding of  how the dataset after converted to timeseries looks like**

In [18]:
wide_window = generate_window(dataset[2])

**Finding the  model's predictions  on any 1  patrol divison**

In [19]:
model = load_model(r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/Trained_model_Files/Multi Head Attention Bi LSTM/mh_attn_bi_lstm_model_files/mh_attn_bi_lstm_all_feature_2.keras')

In [20]:
test_data = wide_window.test

In [21]:

# Unbatch and convert to NumPy arrays
x_test_list = []
y_test_list = []

for x, y in test_data.unbatch():
    x_test_list.append(x.numpy())
    y_test_list.append(y.numpy())

# Convert lists to NumPy arrays
x_test = np.array(x_test_list)
y_test = np.array(y_test_list)

print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

x_test shape: (4381, 24, 16)
y_test shape: (4381, 24, 10)


In [22]:
# Get predicted y values
y_pred = model.predict(x_test)

print("y_pred shape:", y_pred.shape)


137/137 ━━━━━━━━━━━━━━━━━━━━ 12s 53ms/step
y_pred shape: (4381, 24, 10)


In [23]:
# Number of samples you want to inspect
num_samples = 5

# Create a list to store the comparison DataFrames
comparison_list = []

# Loop over the first `num_samples`
for i in range(num_samples):
    true_vals = y_test[i]   # shape: (24, 10)
    pred_vals = y_pred[i]   # shape: (24, 10)

    # Column names for better context
    columns = ['1','2','3','4','5','6','7','8','Count','Group 0']  # Adjust if needed

    df_true = pd.DataFrame(true_vals, columns=[f"{col}" for col in columns])
    df_pred = pd.DataFrame(pred_vals, columns=[f"{col}" for col in columns])

    df_both = pd.concat([df_true, df_pred], axis=1)
    comparison_list.append(df_both)

# Combine all comparisons into one DataFrame
full_comparison = pd.concat(comparison_list, axis=0)

# Reset index
full_comparison.reset_index(drop=True, inplace=True)



In [24]:
full_comparison.head()

,1,2,3,4,5,6,7,8,Count,Group 0,1,2,3,4,5,6,7,8,Count,Group 0
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.031393,-0.010890,0.000403,0.019563,0.003667,-0.000356,-0.000567,-0.000256,-0.013403,0.000808
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,-0.044326,-0.001873,-0.000748,0.039716,0.804399,-0.001614,0.000115,-0.000433,0.900210,-0.000028
2,3.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,6.0,0.0,3.042039,0.018290,-0.000378,3.155807,0.053318,-0.002541,0.000731,0.000234,6.297034,0.000258
3,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.019686,-0.010325,0.000133,0.023688,0.004250,0.017261,0.000489,0.000788,0.943373,0.000435
4,0.0,1.0,0.0,1.0,2.0,1.0,0.0,0.0,5.0,0.0,0.016482,0.804042,0.000201,1.002772,2.159877,0.046289,0.000794,0.000734,5.130755,-0.000226
